# EXP50B: Extended Qwen/SVC Fusion Weight Sweep (0.20 to 0.50)

**Parent Baseline**: EXP47 (Qwen3-8B LoRA24 Centred Twin Head, Local: `0.5910`, Kaggle Public LB: `0.620`)
**Context**: Controlled extension of EXP50 to characterize the weight response surface beyond the arbitrary 0.25 threshold.
**Primary Metric**: Macro-averaged Balanced Accuracy (18 Classes)
**Validation Strategy**: Identical 5-Fold Prompt-Disjoint `GroupKFold(groups=prompt_id)` from EXP50.

## Objectives
1. Evaluate 31 weights: $w_{\text{svc}} \in [0.20, 0.21, \dots, 0.50]$ with $w_{\text{qwen}} = 1 - w_{\text{svc}}$.
2. Evaluate both Official Only (`data/train.csv`) and Official + 36k Clean Open Pool.
3. Determine whether the response surface peaks or continues to the boundary ($0.50$).
4. Calculate paired per-example BA contributions and bootstrap confidence intervals against EXP47.


In [ ]:
import os
import sys
import time
import json
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold


In [ ]:
LABELS = [
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'glm-5.3', 'glm-5.3-flash',
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'kimi-k3', 'qwen3.8-max'
]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)


## 1. Input Artifact Hash Verification


In [ ]:
repo_dir = '/home/rj2826/name-that-llm'
train_path = os.path.join(repo_dir, 'data/train.csv')
pool_path = os.path.join(repo_dir, 'data/external/open_pool_clean.parquet')
oof_qwen_path = os.path.join(repo_dir, 'outputs/oof_exp47_qwen_centered_twin.npy')
oof_pool_svc_path = os.path.join(repo_dir, 'outputs/oof_open_pool_svc.npy')
oof_ref_svc_path = os.path.join(repo_dir, 'outputs/oof_reference_svc.npy')

def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

expected = {
    'train.csv': '40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707',
    'open_pool_clean.parquet': '0c5b289e9f5f7c3f439462d09b3d1d7ef4648839db20309e3476066f3982d5c9',
    'oof_exp47_qwen_centered_twin.npy': '647a75f59789f2574bef7578ebf0e5f00eae2a8481724cfd6316bb4199ad22cb',
    'oof_open_pool_svc.npy': 'cd7e0833a1d6bc60048a3944ccb55018522a407ce37edf7ccd0ec288728ee5b6',
    'oof_reference_svc.npy': '7f0aef0bafca28baac27ff328a4802a2ea027990d33c776b43fc4ff68ef3fda9',
}

for name, p in [('train.csv', train_path), ('open_pool_clean.parquet', pool_path),
                ('oof_exp47_qwen_centered_twin.npy', oof_qwen_path),
                ('oof_open_pool_svc.npy', oof_pool_svc_path),
                ('oof_reference_svc.npy', oof_ref_svc_path)]:
    h = compute_sha256(p)
    assert h == expected[name], f'Hash mismatch on {name}!'
    print(f'Verified {name}: {h[:16]}... [PASS]')


## 2. 31-Weight Extended Sweep (0.20 to 0.50)


In [ ]:
train_df = pd.read_csv(train_path)
y_true = train_df['model'].map(L2I).values
prompts = train_df['prompt_id'].values

P_qwen = np.load(oof_qwen_path)
P_svc_pool = np.load(oof_pool_svc_path)
P_svc_ref = np.load(oof_ref_svc_path)
pred_qwen = P_qwen.argmax(axis=1)
base_ba = balanced_accuracy_score(y_true, pred_qwen)

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(train_df, y_true, groups=prompts))

lp_qwen = np.log(np.clip(P_qwen, 1e-12, 1.0))
lp_svc_pool = np.log(np.clip(P_svc_pool, 1e-12, 1.0))
lp_svc_ref = np.log(np.clip(P_svc_ref, 1e-12, 1.0))

weights = np.round(np.arange(0.20, 0.501, 0.01), 2)

results = []
for regime_name, lp_s in [('Official Only', lp_svc_ref), ('Official + 36k Open Pool', lp_svc_pool)]:
    print(f'\n=== {regime_name} ===')
    prev_ba = None
    for w_s in weights:
        w_q = round(1.0 - w_s, 2)
        oof_f = np.zeros_like(P_qwen)
        fold_bas = []
        for tr_idx, va_idx in folds:
            f_lp = w_q * lp_qwen[va_idx] + w_s * lp_s[va_idx]
            p = np.exp(f_lp - f_lp.max(axis=1, keepdims=True))
            p /= p.sum(axis=1, keepdims=True)
            oof_f[va_idx] = p
            fold_bas.append(balanced_accuracy_score(y_true[va_idx], p.argmax(axis=1)))
        
        ba = balanced_accuracy_score(y_true, oof_f.argmax(axis=1))
        f1 = f1_score(y_true, oof_f.argmax(axis=1), average='macro')
        agree_q = np.mean(oof_f.argmax(axis=1) == pred_qwen)
        d_prev = (ba - prev_ba) if prev_ba is not None else 0.0
        prev_ba = ba
        
        results.append({
            'regime': regime_name, 'w_s': w_s, 'w_q': w_q, 'ba': ba, 'f1': f1,
            'delta_exp47': ba - base_ba, 'delta_prev': d_prev,
            'agree_q': agree_q, 'fold_mean': np.mean(fold_bas), 'fold_std': np.std(fold_bas)
        })
        if round(w_s, 2) in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
            print(f'  w_s={w_s:.2f} (w_q={w_q:.2f}) -> BA={ba:.4f} (d_exp47={ba-base_ba:+.4f}, d_prev={d_prev:+.4f}) | AgreeQ={agree_q*100:.1f}%')

df_res = pd.DataFrame(results)


## 3. Statistical Significance Testing vs EXP47


In [ ]:
classes, class_counts = np.unique(y_true, return_counts=True)
sample_weights = np.zeros(len(y_true), dtype=float)
for c, count in zip(classes, class_counts):
    sample_weights[y_true == c] = 1.0 / (len(classes) * count)

unique_prompts = np.unique(prompts)
n_prompts = len(unique_prompts)
rng = np.random.RandomState(42)

for w_s in [0.25, 0.35, 0.50]:
    w_q = round(1.0 - w_s, 2)
    f_lp = w_q * lp_qwen + w_s * lp_svc_pool
    p = np.exp(f_lp - f_lp.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    pred_c = p.argmax(axis=1)
    
    delta_i = sample_weights * ((pred_c == y_true).astype(float) - (pred_qwen == y_true).astype(float))
    prompt_deltas = np.array([np.sum(delta_i[prompts == p_id]) for p_id in unique_prompts])
    
    boot_deltas = [np.sum(prompt_deltas[rng.choice(n_prompts, size=n_prompts, replace=True)]) for _ in range(5000)]
    ci_l = np.percentile(boot_deltas, 2.5)
    ci_h = np.percentile(boot_deltas, 97.5)
    p_pos = np.mean(np.array(boot_deltas) > 0)
    print(f'w_s={w_s:.2f}: Delta={np.sum(delta_i):+.4f} | 95% CI: [{ci_l:+.4f}, {ci_h:+.4f}] | P(delta > 0) = {p_pos:.4f}')


## 4. Key Empirical Findings
- **Boundary Not Resolved**: Both Open Pool and Official-Only curves increase monotonically across the entire range $0.20$ to $0.50$, reaching numerical maxima at $w_{\text{svc}} = 0.50$ (BA: 0.6354 and 0.6147).
- **Surface Shift Caution**: In EXP49, local CV peaked at $w_{\text{svc}}=0.85$ (BA 0.7028) but failed on Kaggle (0.612). Local CV's upward pull toward high SVC weights is driven by the $8.24\times$ logit scale difference and training prompt density.
- **Candidate Selection**: While $w_{\text{svc}}=0.50$ is the local CV maximum in $[0.20, 0.50]$, conservative candidates ($w_{\text{svc}}=0.25$ or $0.35$) preserve $>95\%$ Qwen agreement and provide safer hedges against unseen Kaggle prompt shift.
